# Notebook 01b: Interactive First Look

Interactive charts with Plotly. Hover to explore, toggle fuel types on/off.

Also: color palette options and normalized (%) view.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df24 = pd.read_csv('../data/processed/india_2024.csv', parse_dates=['date'])

# The 8 fuel types we care about (in stacking order, bottom to top)
fuel_cols = ['coal', 'lignite', 'gas', 'nuclear', 'hydro', 'wind', 'solar', 'other_re']
fuel_labels = ['Coal', 'Lignite', 'Gas', 'Nuclear', 'Hydro', 'Wind', 'Solar', 'Other RE']

## Color Palette Options

Three palettes to choose from — each designed to be **representative** of the source:

In [2]:
# === PALETTE A: "Earth & Sky" ===
# Coal/lignite = dark earthy tones (what it IS — mined from the ground)
# Gas = blue flame
# Nuclear = radioactive green-teal
# Hydro = deep water blue
# Wind = light sky blue
# Solar = warm golden
palette_a = {
    'coal': '#3D2B1F',      # dark brown (earth)
    'lignite': '#6B4226',   # lighter brown
    'gas': '#4A90D9',       # blue flame
    'nuclear': '#2EC4B6',   # teal-green (radioactive glow)
    'hydro': '#1B4F72',     # deep water blue
    'wind': '#85C1E9',      # sky blue
    'solar': '#F5B041',     # golden sun
    'other_re': '#A569BD',  # purple (biomass/misc)
}

# === PALETTE B: "Industrial Truth" ===
# Coal = charcoal black-grey (honest about what it is)
# Gas = steel blue
# Nuclear = warning amber
# Renewables = natural greens and yellows
palette_b = {
    'coal': '#2C2C2C',      # charcoal
    'lignite': '#5C5C5C',   # grey
    'gas': '#5B8DBE',       # steel blue
    'nuclear': '#E67E22',   # amber/warning
    'hydro': '#2471A3',     # river blue
    'wind': '#27AE60',      # fresh green
    'solar': '#F1C40F',     # bright yellow
    'other_re': '#8E44AD',  # purple
}

# === PALETTE C: "Warm Spectrum" ===
# Fossil fuels = warm reds/oranges (heat, combustion)
# Clean sources = cool blues/greens
# Solar = stands out in gold
palette_c = {
    'coal': '#922B21',      # deep red (burning)
    'lignite': '#C0392B',   # red
    'gas': '#E74C3C',       # lighter red-orange
    'nuclear': '#1ABC9C',   # clean teal
    'hydro': '#2980B9',     # blue
    'wind': '#27AE60',      # green
    'solar': '#F39C12',     # gold
    'other_re': '#8E44AD',  # purple
}

palettes = {'A: Earth & Sky': palette_a, 'B: Industrial Truth': palette_b, 'C: Warm Spectrum': palette_c}

In [3]:
# Show all 3 palettes as color swatches
fig = make_subplots(rows=3, cols=1, subplot_titles=list(palettes.keys()),
                    vertical_spacing=0.12)

for i, (name, pal) in enumerate(palettes.items(), 1):
    for j, fuel in enumerate(fuel_cols):
        fig.add_trace(go.Bar(
            x=[fuel_labels[j]], y=[1],
            marker_color=pal[fuel],
            name=fuel_labels[j],
            showlegend=(i == 1),
            text=pal[fuel], textposition='inside',
            textfont=dict(color='white', size=10),
        ), row=i, col=1)

fig.update_layout(height=500, title='Color Palette Options', barmode='group',
                  plot_bgcolor='#FAFAF5')
fig.update_yaxes(visible=False)
fig.show()

## Interactive Stacked Area — All 3 Palettes

Hover over any point to see the exact generation by fuel type. Click legend items to toggle.

In [4]:
def make_stacked_area(df, palette, title, normalize=False):
    """Create interactive stacked area chart."""
    # 7-day rolling average
    smooth = df.set_index('date')[fuel_cols].rolling(7, center=True).mean().dropna()
    
    if normalize:
        row_totals = smooth.sum(axis=1)
        smooth = smooth.div(row_totals, axis=0) * 100
    
    fig = go.Figure()
    for col, label in zip(fuel_cols, fuel_labels):
        fig.add_trace(go.Scatter(
            x=smooth.index, y=smooth[col],
            name=label,
            mode='lines',
            line=dict(width=0.5, color=palette[col]),
            fillcolor=palette[col],
            stackgroup='one',
            groupnorm='percent' if False else '',  # we handle normalization ourselves
            hovertemplate=f'{label}: ' + ('%{y:.1f}%' if normalize else '%{y:.0f} MU') + '<extra></extra>',
        ))
    
    y_title = 'Share of Generation (%)' if normalize else 'Generation (MU = GWh)'
    fig.update_layout(
        title=title,
        xaxis_title='',
        yaxis_title=y_title,
        hovermode='x unified',
        plot_bgcolor='#FAFAF5',
        paper_bgcolor='#FAFAF5',
        height=500,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    )
    return fig

In [5]:
# Palette A: Earth & Sky
fig_a = make_stacked_area(df24, palette_a, 'Palette A: Earth & Sky — India Grid 2024 (MU/day, 7-day avg)')
fig_a.show()

In [6]:
# Palette B: Industrial Truth
fig_b = make_stacked_area(df24, palette_b, 'Palette B: Industrial Truth — India Grid 2024 (MU/day, 7-day avg)')
fig_b.show()

In [7]:
# Palette C: Warm Spectrum
fig_c = make_stacked_area(df24, palette_c, 'Palette C: Warm Spectrum — India Grid 2024 (MU/day, 7-day avg)')
fig_c.show()

## Normalized View — Generation Mix as Percentage

This strips out the absolute demand growth and shows **what fraction of electricity comes from each source** day by day. Watch how the monsoon reshuffles the deck.

In [8]:
# Normalized with each palette
fig_a_norm = make_stacked_area(df24, palette_a, 'Palette A — Normalized: Generation Mix % (7-day avg)', normalize=True)
fig_a_norm.show()

In [9]:
fig_b_norm = make_stacked_area(df24, palette_b, 'Palette B — Normalized: Generation Mix % (7-day avg)', normalize=True)
fig_b_norm.show()

In [10]:
fig_c_norm = make_stacked_area(df24, palette_c, 'Palette C — Normalized: Generation Mix % (7-day avg)', normalize=True)
fig_c_norm.show()

## The Solar Dip — Monsoon Cloud Cover

The gap you noticed in solar is real. Let's isolate it.

In [11]:
# Solar, Wind, Hydro — the monsoon trio
fig_monsoon = go.Figure()

for col, label, color in [
    ('solar', 'Solar', '#F5B041'),
    ('wind', 'Wind', '#27AE60'),
    ('hydro', 'Hydro', '#2471A3'),
]:
    smooth = df24.set_index('date')[col].rolling(7, center=True).mean()
    fig_monsoon.add_trace(go.Scatter(
        x=smooth.index, y=smooth,
        name=label, mode='lines',
        line=dict(width=2.5, color=color),
        hovertemplate=f'{label}: %{{y:.0f}} MU<extra></extra>',
    ))

# Mark monsoon season
fig_monsoon.add_vrect(x0='2024-06-01', x1='2024-09-30',
                       fillcolor='rgba(100,149,237,0.1)', line_width=0,
                       annotation_text='Monsoon Season', annotation_position='top left')

fig_monsoon.update_layout(
    title='The Monsoon Effect: Solar Dips, Hydro & Wind Surge',
    yaxis_title='Generation (MU/day, 7-day avg)',
    hovermode='x unified',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=450,
)
fig_monsoon.show()

In [12]:
# Coal vs Renewables — the tension
smooth = df24.set_index('date')[['coal', 'renewables']].rolling(7, center=True).mean()

fig_tension = go.Figure()
fig_tension.add_trace(go.Scatter(x=smooth.index, y=smooth['coal'],
    name='Coal', line=dict(width=2.5, color='#3D2B1F'),
    hovertemplate='Coal: %{y:.0f} MU<extra></extra>'))
fig_tension.add_trace(go.Scatter(x=smooth.index, y=smooth['renewables'],
    name='Renewables (Wind+Solar+Other)', line=dict(width=2.5, color='#27AE60'),
    hovertemplate='Renewables: %{y:.0f} MU<extra></extra>'))

fig_tension.update_layout(
    title='The Tension: Coal vs Total Renewables (2024)',
    yaxis_title='Generation (MU/day, 7-day avg)',
    hovermode='x unified',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=450,
)
fig_tension.show()

## Summary

**Solar dip explained:** Monsoon cloud cover (Jul-Aug) drops solar from ~380 MU/day to ~300 MU/day. But nature compensates — hydro triples and wind doubles during the same period.

**Normalized view reveals:** Coal's *share* drops from ~75% in winter to ~65% during monsoon, even though absolute coal generation stays high. The monsoon is when India's grid is cleanest.

**Pick your palette** — which one tells the story best?